In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Numerical preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both pipelines
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Preprocessing pipeline created successfully!")
print(preprocessor)

Preprocessing pipeline created successfully!
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['pclass', 'age', 'sibsp', 'parch', 'fare']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['sex', 'embarked', 'class', 'who', 'deck',
                                  'embark_town', 'alive'])])


In [5]:
# Get names of the transformed features

feature_names = preprocessor.get_feature_names_out()

print("Total processed features:", len(feature_names))

print("\nFirst 20 processed features:")
for feature in feature_names[:20]:
    print(feature)

NotFittedError: This ColumnTransformer instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In [6]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)

print("Original testing shape:", X_test.shape)
print("Processed testing shape:", X_test_processed.shape)

Original training shape: (712, 14)
Processed training shape: (712, 28)
Original testing shape: (179, 14)
Processed testing shape: (179, 28)


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

model.fit(X_train, y_train)

print("Random Forest model trained successfully!")

Random Forest model trained successfully!


In [8]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Make predictions on test data
y_pred = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Test Accuracy:", round(accuracy, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Test Accuracy: 1.0

Confusion Matrix:
[[110   0]
 [  0  69]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       110
           1       1.00      1.00      1.00        69

    accuracy                           1.00       179
   macro avg       1.00      1.00      1.00       179
weighted avg       1.00      1.00      1.00       179



In [9]:
# Feature Importance Analysis

trained_preprocessor = model.named_steps["preprocessor"]
trained_classifier = model.named_steps["classifier"]

feature_names = trained_preprocessor.get_feature_names_out()
importances = trained_classifier.feature_importances_

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False)

print("Top 10 Most Important Features:")
print(feature_importance.head(10).to_string(index=False))

Top 10 Most Important Features:
         Feature  Importance
  cat__alive_yes    0.438216
   cat__alive_no    0.338422
   cat__sex_male    0.048629
 cat__sex_female    0.041799
    cat__who_man    0.034634
  cat__who_woman    0.025355
       num__fare    0.012192
        num__age    0.010728
cat__class_Third    0.009895
     cat__deck_C    0.008345


In [10]:
# Correlation Analysis

print("Numerical Feature Correlation:")
print(X_train[numeric_features].corr().round(2))

print("\n--- Task 03 Summary ---")
print("1. Train/test split performed before preprocessing.")
print("2. Numerical features: median imputation + StandardScaler.")
print("3. Categorical features: most-frequent imputation + OneHotEncoder.")
print("4. ColumnTransformer used to combine preprocessing steps.")
print("5. Random Forest classifier integrated into the ML pipeline.")
print("6. Feature importance analysis completed.")
print("7. Test-set evaluation completed.")
print("8. Preprocessing was fitted only on training data to reduce data leakage.")

Numerical Feature Correlation:
        pclass   age  sibsp  parch  fare
pclass    1.00 -0.35   0.10   0.04 -0.56
age      -0.35  1.00  -0.31  -0.18  0.11
sibsp     0.10 -0.31   1.00   0.39  0.13
parch     0.04 -0.18   0.39   1.00  0.18
fare     -0.56  0.11   0.13   0.18  1.00

--- Task 03 Summary ---
1. Train/test split performed before preprocessing.
2. Numerical features: median imputation + StandardScaler.
3. Categorical features: most-frequent imputation + OneHotEncoder.
4. ColumnTransformer used to combine preprocessing steps.
5. Random Forest classifier integrated into the ML pipeline.
6. Feature importance analysis completed.
7. Test-set evaluation completed.
8. Preprocessing was fitted only on training data to reduce data leakage.


In [3]:
# Identify numerical and categorical columns

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numerical Features:")
print(numeric_features)

print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
['pclass', 'age', 'sibsp', 'parch', 'fare']

Categorical Features:
['sex', 'embarked', 'class', 'who', 'deck', 'embark_town', 'alive']


C:\Users\mural\AppData\Local\Temp\ipykernel_2916\556807264.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()


In [2]:
from sklearn.model_selection import train_test_split

X = df.drop("survived", axis=1)
y = df["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 712
Testing samples: 179


In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"

df = pd.read_csv(url)

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

Dataset Shape: (891, 15)

Columns:
['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']

Missing Values:
survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64
